# Homework 2: Vector Search

Setup first:
```bash
# mkdir [your code directory] #Uncomment this!
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
```

Download two helper scripts from the `embed/` directory of the course repo:
- Option 1:
```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py
```
- Option 2, because Windows does not include the wget utility by default:
```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
curl -O $PREFIX/download.py
curl -O $PREFIX/embedder.p
```

Run `download.py` to fetch `Xenova/all-MiniLM-L6-v2`, the ONNX version of the `all-MiniLM-L6-v2` model from the lessons:

```bash
uv run python download.py
```

In [1]:
# Test notebook
print("Hello World!")

Hello World!


## Q1. Embedding a query

In [5]:
import numpy as np
from embedder import Embedder

# Initialize the embedder
# Ensure the model has been downloaded to the default 'models' folder using download.py
embedder = Embedder(path="models/Xenova/all-MiniLM-L6-v2")

# Define the query from the homework
query = "How does approximate nearest neighbor search work?"

# Embed the query to get the vector
v = embedder.encode(query)

In [6]:
# Confirm the shape
print(v.shape)

(384,)


In [7]:
# Confirm v got normalized
print(np.linalg.norm(v))

1.0


In [10]:
# Print the first value (v)
print(f"The first value (v) is: {v[0]:.2f}")

The first value (v) is: -0.02


## Q2. Cosine similarity

In [11]:
# Initialize the reader to fetch markdown files from the /lessons/ folders
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [12]:
# Check modules name
modules = set(d["filename"].split("/")[0] for d in documents)
print(modules)

{'05-monitoring', '03-orchestration', '06-best-practices', '07-project-example', '01-agentic-rag', '04-evaluation', '02-vector-search'}


In [14]:
# Count the number of lesson pages (documents)
print(f"Number of lesson pages: {len(documents)}")

Number of lesson pages: 72


In [15]:
target_file = '02-vector-search/lessons/07-sqlitesearch-vector.md'
target = [d for d in documents if d["filename"] == target_file][0]
print(target)

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [17]:
page_vec = embedder.encode(target["content"])
print(np.linalg.norm(page_vec))

0.9999999999999999


In [20]:
# Cosine similarity
similarity = page_vec.dot(v)  # v is our Q1 query vector
print(f"The cosine similarity is: {similarity:.3f}")

The cosine similarity is: 0.361


## Q3. Chunking and search by hand

In [21]:
# Import library
from gitsource import chunk_documents

# Apply chunking with a 2000-character window and 1000-character step
# This will split the 72 pages into smaller pieces
chunks = chunk_documents(documents, size=2000, step=1000)

# Count the total number of resulting chunks
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [22]:
# Embed every chunk and stack them into a matrix X
chunk_texts = [c["content"] for c in chunks]
X = embedder.encode_batch(chunk_texts)

# Check X shape
print(X.shape)

(295, 384)


In [26]:
# Confirm X got normalized
print(np.linalg.norm(X, axis=1))

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1.]


In [28]:
# Compute similarity scores
scores = X.dot(v)

In [29]:
# Evaluate the top 5, not just the top 1
top_k = scores.argsort()[::-1][:5]

for i in top_k:
    print(chunks[i]["filename"], scores[i])

02-vector-search/lessons/07-sqlitesearch-vector.md 0.6489017718578813
01-agentic-rag/lessons/05-search.md 0.5510346348435939
04-evaluation/lessons/05-search-metrics.md 0.4065607072060337
02-vector-search/lessons/04-vector-search.md 0.40618197902402514
06-best-practices/lessons/03-reranking.md 0.40610594157880786


In [31]:
# Find the filename of the highest-scoring chunk
best_idx = scores.argmax()
best_chunk = chunks[best_idx]

print(f"Highest Score: {scores[best_idx]:.3f}")
print(f"Filename: {best_chunk['filename']}")

Highest Score: 0.649
Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch

In [32]:
from minsearch import VectorSearch

# Initialize VectorSearch
vector_index = VectorSearch(keyword_fields=["filename"])

# Index the chunks and their vectors
# X is the matrix of embeddings from Q3; chunks is the list of chunk dictionaries
vector_index.fit(X, chunks)

In [33]:
# Embed the new homework query
query2 = "What metric do we use to evaluate a search engine?"
v2 = embedder.encode(query2)

In [35]:
# Confirm the shape and normalization
print(f"Shape : {v2.shape}")
print(f"Normalized result : {np.linalg.norm(v2)}")

Shape : (384,)
Normalized result : 0.9999999999999999


In [42]:
# Perform the vector search for top 5 result
results = vector_index.search(v2, num_results=5)
for i in results:
    print(i["filename"])

print(f"\nThe first result for this query: {results[0]['filename']}")

04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/01-intro.md
01-agentic-rag/lessons/05-search.md
04-evaluation/lessons/01-intro.md
04-evaluation/lessons/15-next-steps.md

The first result for this query: 04-evaluation/lessons/05-search-metrics.md


## Q5. Text search vs vector search

In [43]:
from minsearch import Index

# Setup the Text (Keyword) Index
# Using 'content' as the searchable text field
text_index = Index(
    text_fields=["content"], 
    keyword_fields=["filename"]
)
text_index.fit(chunks)

In [44]:
# Define the query and encode it for vector search
query3 = "How do I store vectors in PostgreSQL?"
v3 = embedder.encode(query3)

# Confirm the shape and normalization
print(f"Shape : {v3.shape}")
print(f"Normalized resulst : {np.linalg.norm(v3)}")

Shape : (384,)
Normalized resulst : 0.9999999999999999


In [45]:
# Perform both searches
vector_results = vector_index.search(v3, num_results=5)
text_results = text_index.search(query3, num_results=5)

# Compare the resulting filenames
vector_filenames = {r["filename"] for r in vector_results}
text_filenames = {r["filename"] for r in text_results}

print("Vector Search Top 5:", vector_filenames)
print("Text Search Top 5:", text_filenames)

Vector Search Top 5: {'03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md'}
Text Search Top 5: {'02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md'}


In [47]:
# Find the file in Vector results but not Text results
only_in_vector = vector_filenames - text_filenames
print(f"File in vector but not text: {only_in_vector}")

File in vector but not text: {'02-vector-search/lessons/08-pgvector.md'}


## Q6. Hybrid search

In [48]:
# Define the RRF function provided in the homework
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# Execute searches for the query
query4 = "How do I give the model access to tools?"
v4 = embedder.encode(query4)

In [49]:
# Confirm the shape and normalization
print(f"Shape : {v4.shape}")
print(f"Normalized resulst : {np.linalg.norm(v4)}")

Shape : (384,)
Normalized resulst : 1.0


In [50]:
# Perform both searches with Top 5 results
text_results_q6 = text_index.search(query4, num_results=5)
vector_results_q6 = vector_index.search(v4, num_results=5)

# Fuse results (text, vector)
hybrid_results = rrf([text_results_q6, vector_results_q6])

In [53]:
# Top 5 results
for i in hybrid_results: 
    print(i["filename"])

# Output the filename of the first result
print(f"\nTop Hybrid Result: {hybrid_results[0]['filename']}")

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/13-function-calling.md

Top Hybrid Result: 01-agentic-rag/lessons/13-function-calling.md


Thanks!!!